<a href="https://colab.research.google.com/github/zuykovmikhail1305-boop/masterclass/blob/main/agents_workshop_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ИИ-агенты на практике: tools, LangChain, LangGraph и MCP

Сегодня мы соберем агента по шагам:

1. Сначала посмотрим, чем обычная LLM отличается от агента.
2. Напишем свои инструменты как обычные Python-функции.
3. Подключим их к агенту через LangChain `create_agent`.
4. Соберем тот же agent loop явно через LangGraph.
5. Подключим готовый web-сервис You.com через MCP.
6. В конце вы напишете свой tool для поиска по мини-базе знаний.

## 0. Установка библиотек

Colab запускается в чистом окружении, поэтому сначала ставим все нужные пакеты.

In [ ]:
%pip install -q -U langchain langchain-openrouter langchain-experimental langgraph langchain-mcp-adapters mcp httpx nest_asyncio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

## 1. Настройка OpenRouter

Нужен `OPENROUTER_API_KEY`. Для free tier в модели используем `openrouter/free`.

Если вы уже ввели неправильный ключ, поставьте `RESET_OPENROUTER_API_KEY = True` в следующей ячейке и запустите ее заново.

In [ ]:
import os
import getpass
import nest_asyncio

nest_asyncio.apply()

OPENROUTER_API_KEY = getpass.getpass("Введите OPENROUTER_API_KEY: ")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

Введите OPENROUTER_API_KEY: ··········


In [ ]:
from langchain_openrouter import ChatOpenRouter

MODEL_NAME = "openrouter/free"

model = ChatOpenRouter(
    model=MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    temperature=0,
    app_title="Agent Workshop",
)

print("Модель готова:", MODEL_NAME)


Модель готова: openrouter/free


## 2. Обычная LLM

Обычная модель получает текст и возвращает текст. Она может рассуждать, но сама не вызывает Python-функции и не ходит в API.

In [ ]:
answer = model.invoke("Объясни школьнику за 2 предложения, что такое API.")
print(answer.content)

Представь, что API — это официант в ресторане: ты (программа) говоришь ему, что хочешь заказать, а он приносит готовое блюдо (данные или функцию) с кухни (другого сервера), не заставляя тебя лезть за плиту и готовить самому. Это набор правил, по которым одна программа просит у другой выполнить работу и получить результат, не зная, как именно это устроено внутри.


## 3. Tools: обычные функции

Tool - это обычная функция, но с понятным названием, типами аргументов и описанием. По этому описанию модель понимает, когда инструмент нужно вызвать.

In [ ]:
from pathlib import Path

from langchain_core.tools import tool
from langchain_experimental.tools.python.tool import PythonREPLTool

WORKSPACE = Path("agent_workspace")
WORKSPACE.mkdir(exist_ok=True)


def safe_path(filename: str) -> Path:
    """Return a safe file path inside the agent workspace."""
    clean_name = Path(filename).name
    return WORKSPACE / clean_name


@tool
def write_text_file(filename: str, content: str) -> str:
    """Write a text file into the agent workspace."""
    path = safe_path(filename)
    path.write_text(content, encoding="utf-8")
    return f"Файл {path} создан. Размер: {len(content)} символов."


@tool
def read_text_file(filename: str) -> str:
    """Read a text file from the agent workspace."""
    path = safe_path(filename)
    if not path.exists():
        return f"Файл {path} не найден."
    return path.read_text(encoding="utf-8")


@tool
def list_workspace_files() -> str:
    """List files in the agent workspace."""
    files = sorted(path.name for path in WORKSPACE.iterdir() if path.is_file())
    if not files:
        return "В рабочей папке пока нет файлов."
    return "Файлы в рабочей папке:\n" + "\n".join(files)


# Такой инструмент часто не пишут руками: его уже сделали в библиотеке.
# Важно: Python REPL выполняет код, поэтому в реальных проектах его дают агенту только в sandbox.
python_repl = PythonREPLTool(
    name="python_repl",
    description="Run Python code for calculations. Use print(...) to show the result.",
)


tools = [write_text_file, read_text_file, list_workspace_files, python_repl]

for tool_item in tools:
    print(tool_item.name, "->", tool_item.description)

write_text_file -> Write a text file into the agent workspace.
read_text_file -> Read a text file from the agent workspace.
list_workspace_files -> List files in the agent workspace.
python_repl -> Run Python code for calculations. Use print(...) to show the result.


Пока это просто Python. Инструмент можно вызвать вручную.

In [ ]:
print(
    write_text_file.invoke(
        {
            "filename": "hello.txt",
            "content": "Привет! Этот файл создал Python tool.",
        }
    )
)
print(list_workspace_files.invoke({}))
print(read_text_file.invoke({"filename": "hello.txt"}))
print(
    python_repl.invoke(
        {"query": "numbers = [5, 4, 3, 5]\nprint(sum(numbers) / len(numbers))"}
    )
)

Файл agent_workspace/hello.txt создан. Размер: 37 символов.
Файлы в рабочей папке:
hello.txt
Привет! Этот файл создал Python tool.
4.25



## 4. LangChain `create_agent`

Теперь отдадим tools агенту. Агент сам решит, какие функции нужно вызвать.

In [ ]:
from langchain.agents import create_agent

study_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "Ты учебный ИИ-агент для школьников. "
        "Ты умеешь работать с файлами в agent_workspace и использовать готовый Python REPL tool для вычислений. "
        "Если для ответа нужно действие в компьютере, используй инструмент. "
        "Отвечай по-русски, коротко и понятно."
    ),
)

result = study_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Создай файл git_plan.txt с коротким планом подготовки по Git. "
                    "Потом покажи список файлов, прочитай этот файл и с помощью Python REPL "
                    "посчитай среднее для оценок 5, 4, 3, 5."
                ),
            }
        ]
    }
)

final_content = result["messages"][-1].content
if isinstance(final_content, list):
    print(final_content[0].get("text", final_content))
else:
    print(final_content)

Готово! Вот что получилось:

1. **Файл `git_plan.txt`** создан с кратким планом подготовки по Git.
2. **Список файлов** в рабочей папке: `git_plan.txt`, `hello.txt`.
3. **Содержимое файла** прочитано (план из 7 пунктов).
4. **Среднее для оценок 5, 4, 3, 5** = **4.25**.


`create_agent` удобен, когда хочется быстро получить готового агента. Но подробную механику tool calling удобнее смотреть в LangGraph, где мы сами задаем узел модели и узел tools.

## 5. LangGraph: собираем agent loop явно

`create_agent` удобен, но он скрывает workflow. В LangGraph мы сами задаем узлы и переходы.

Минимальная схема:

```text
START -> agent -> tools -> agent -> END
```

In [ ]:
import json
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


def message_text(message) -> str:
    """Convert LangChain message content to readable text."""
    content = message.content
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(item.get("text") or str(item))
            else:
                parts.append(str(item))
        return "\n".join(part for part in parts if part)
    return str(content)


def print_agent_trace(messages):
    """Show messages and structured tool calls from the LangGraph agent loop."""
    for i, message in enumerate(messages, start=1):
        print("=" * 90)
        print(f"{i}. {message.type}")

        text = message_text(message).strip()
        if text:
            print("content:")
            print(text)

        tool_calls = getattr(message, "tool_calls", None) or []
        if tool_calls:
            print("tool_calls generated by model:")
            for call in tool_calls:
                print(
                    json.dumps(
                        {
                            "name": call.get("name"),
                            "args": call.get("args"),
                            "id": call.get("id"),
                        },
                        ensure_ascii=False,
                        indent=2,
                    )
                )

        if message.type == "tool":
            print("tool result metadata:")
            print(
                json.dumps(
                    {
                        "name": getattr(message, "name", None),
                        "tool_call_id": getattr(message, "tool_call_id", None),
                    },
                    ensure_ascii=False,
                    indent=2,
                )
            )


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


model_with_tools = model.bind_tools(tools)


def agent_node(state: AgentState):
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}


graph_builder = StateGraph(AgentState)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", ToolNode(tools))

graph_builder.add_edge(START, "agent")
graph_builder.add_conditional_edges("agent", tools_condition)
graph_builder.add_edge("tools", "agent")

study_graph = graph_builder.compile()

print("LangGraph готов")

LangGraph готов


In [ ]:
graph_result = study_graph.invoke(
    {
        "messages": [
            SystemMessage(
                content=(
                    "Ты учебный агент. Используй tools, когда нужно выполнить действие "
                    "в компьютере. Отвечай по-русски."
                )
            ),
            HumanMessage(
                content="Создай файл api_notes.txt с 3 фактами про API, прочитай его и покажи список файлов."
            ),
        ]
    }
)

final_content = graph_result["messages"][-1].content
if isinstance(final_content, list):
    print(final_content[0].get("text", final_content))
else:
    print(final_content)



Готово! Все задачи выполнены:

**1. Создан файл api_notes.txt** с 3 фактами про API:
- Факт 1: API (Application Programming Interface) — это набор правил и протоколов, позволяющих различным программным компонентам взаимодействовать друг с другом.
- Факт 2: REST (Representational State Transfer) — это архитектурный стиль для создания веб-сервисов, использующих HTTP-протоколы для передачи данных.
- Факт 3: API ключи используются для аутентификации и авторизации доступа к API, обеспечивая безопасность и контроль над использованием сервисов.

**2. Файл прочитан** — содержимое подтверждено (465 символов).

**3. Список файлов в рабочей папке:**
- api_notes.txt
- git_plan.txt
- hello.txt


In [ ]:
print_agent_trace(graph_result["messages"])

1. system
content:
Ты учебный агент. Используй tools, когда нужно выполнить действие в компьютере. Отвечай по-русски.
2. human
content:
Создай файл api_notes.txt с 3 фактами про API, прочитай его и покажи список файлов.
3. ai
tool_calls generated by model:
{
  "name": "write_text_file",
  "args": {
    "filename": "api_notes.txt",
    "content": "Факт 1: API (Application Programming Interface) - это набор правил и протоколы, которые позволяют различным программным компонентам взаимодействовать друг с другом.\n\nФакт 2: REST (Representational State Transfer) - это архитектурный стиль для создания веб-сервисов, которые используют HTTP-протоколы для передачи данных.\n\nФакт 3: API ключи используются для аутентификации и авторизации доступа к API, обеспечивая безопасность и контроль над использованием сервисов."
  },
  "id": "write_text_file_qz116jvkm7c1"
}
4. tool
content:
Файл agent_workspace/api_notes.txt создан. Размер: 465 символов.
tool result metadata:
{
  "name": "write_text_file",

## 6. MCP: подключаем You.com как готовый web-сервис

Теперь подключаем не нашу функцию и не самописный сервер, а готовый MCP-сервис.

You.com MCP дает агенту набор инструментов для работы с вебом: поиск, извлечение содержимого страниц и research-ответы. Мы не пишем эти tools руками, а просто подключаем внешний сервис через MCP.

Подключение готового MCP-сервера. Главная часть здесь - конфиг: команда запуска `npx @youdotcom-oss/mcp` и переменные окружения с ключом.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import langchain_mcp_adapters.sessions as lc_sessions
from mcp.client.stdio import stdio_client as _orig_stdio_client
from google.colab import userdata

# В Colab sys.stderr не имеет fileno(), а stdio MCP transport его требует.
# Поэтому логи MCP-сервера отправляем в обычный файл.
mcp_log = open("/tmp/mcp_server.log", "w")

def _stdio_client_colab(server, errlog=None):
    return _orig_stdio_client(server, errlog=mcp_log)

# Подменяем функцию там, где её вызывает langchain_mcp_adapters
lc_sessions.stdio_client = _stdio_client_colab

you_client = MultiServerMCPClient(
    {
        "you": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@youdotcom-oss/mcp"],
            "env": {
                "YDC_API_KEY": userdata.get("you"),
                "YDC_ALLOWED_TOOLS": "you-search,you-research",
            },
        }
    }
)

you_tools = await you_client.get_tools()

print(f"Получили tools от You.com MCP: {len(you_tools)}")
for you_tool in you_tools:
    print("-", you_tool.name, "->", you_tool.description[:140])

Получили tools от You.com MCP: 2
- you-search -> Search the web for current information, facts, news, and sources.
>
Best for: current events, facts about people, companies, and organizatio
- you-research -> One-shot cited synthesis — research a topic and return a concise answer with inline citations and source list. Use when a synthesized, sourc


Теперь используем You.com MCP tools в LangGraph. Граф остается тем же: модель генерирует `tool_calls`, `ToolNode` выполняет tools, результат возвращается модели.

In [ ]:
you_model_with_tools = model.bind_tools(you_tools)


async def you_agent_node(state: AgentState):
    response = await you_model_with_tools.ainvoke(state["messages"])
    return {"messages": [response]}


you_graph_builder = StateGraph(AgentState)
you_graph_builder.add_node("agent", you_agent_node)
you_graph_builder.add_node("tools", ToolNode(you_tools, handle_tool_errors=True))

you_graph_builder.add_edge(START, "agent")
you_graph_builder.add_conditional_edges("agent", tools_condition)
you_graph_builder.add_edge("tools", "agent")

you_graph = you_graph_builder.compile()

you_result = await you_graph.ainvoke(
    {
        "messages": [
            SystemMessage(
                content=(
                    "Ты агент, который использует You.com MCP tools для поиска и анализа веб-страниц. "
                    "Когда нужна актуальная информация из интернета, используй доступные tools. "
                    "Отвечай по-русски, кратко и с опорой на найденные источники."
                )
            ),
            HumanMessage(
                content=(
                    "Сделай web research про LangGraph. "
                    "Найди свежую информацию и кратко объясни школьнику, зачем нужен LangGraph. "
                    "Если один инструмент вернул ошибку, используй другой доступный инструмент."
                )
            ),
        ]
    }
)

final_content = you_result["messages"][-1].content
if isinstance(final_content, list):
    print(final_content[0].get("text", final_content))
else:
    print(final_content)

In [ ]:
print_agent_trace(you_result["messages"])

1. system
content:
Ты агент, который использует You.com MCP tools для поиска и анализа веб-страниц. Когда нужна актуальная информация из интернета, используй доступные tools. Отвечай по-русски, кратко и с опорой на найденные источники.
2. human
content:
Сделай web research про LangGraph. Найди свежую информацию и кратко объясни школьнику, зачем нужен LangGraph. Если один инструмент вернул ошибку, используй другой доступный инструмент.
3. ai
tool_calls generated by model:
{
  "name": "you-research",
  "args": {
    "input": "Что такое LangGraph и зачем он нужен? Найди свежую информацию о LangGraph, его назначении, принципах работы и преимуществах по сравнению с другими фреймворками для разработки агентов и рабочих процессов на основе языковых моделей.",
    "research_effort": "standard"
  },
  "id": "call_9cc0e2cc9f164b20ae748a65"
}
4. tool
content:
Research task queued (task_id: "3529d456-d0ca-4b75-b080-111396fa26b8"). Call this tool again with task_id="3529d456-d0ca-4b75-b080-111396fa

## 7. Практика: свой поиск по базе знаний

Теперь ваша очередь. Ниже есть маленькая база учебных материалов. Нужно написать tool `search_materials`, подключить его к агенту и проверить вопросы.

Это не MCP. Это ваш собственный Python tool.

In [ ]:
MATERIALS = [
    {
        "title": "Git: история проекта без паники",
        "tags": ["git", "github", "version control"],
        "text": "Git помогает сохранять версии кода, создавать ветки и возвращаться к старым изменениям.",
    },
    {
        "title": "API: как программы общаются",
        "tags": ["api", "backend", "http"],
        "text": "API задает правила, по которым одна программа может попросить данные или действие у другой программы.",
    },
    {
        "title": "Python-функции",
        "tags": ["python", "functions", "code"],
        "text": "Функция получает аргументы, выполняет код и возвращает результат. Tools агента тоже часто являются функциями.",
    },
    {
        "title": "Нейросети и LLM",
        "tags": ["ai", "llm", "neural networks"],
        "text": "LLM анализирует текст и генерирует ответ, но без tools не выполняет действия во внешнем мире.",
    },
    {
        "title": "Кибербезопасность: базовая гигиена",
        "tags": ["security", "passwords", "2fa"],
        "text": "Сильные пароли, двухфакторная аутентификация и осторожность с ссылками снижают риск взлома.",
    },
]

print("Материалов в базе:", len(MATERIALS))